# Embedding Training

Fine-tune `sentence-transformers/all-MiniLM-L6-v2` with MultipleNegativesRankingLoss on NFCorpus triplets.

**Data:** `triplets.jsonl` — NFCorpus train qrels + BM25 hard negatives  
**Loss:** MultipleNegativesRankingLoss — contrasts query against all other positives in batch  
**Output:** `embedding_domain/` on Drive  
**Eval:** Triplet accuracy (positive cosine > negative)

1. Upload `data/en/triplets.jsonl` to Drive: `MyDrive/nutrition-rag/triplets.jsonl`
2. Run cells in order

In [ ]:
!pip install -q sentence-transformers datasets

## Setup

In [ ]:
import os
from google.colab import drive

drive.mount("/content/drive")

DRIVE_BASE = "/content/drive/MyDrive/nutrition-rag"
DATA_PATH  = f"{DRIVE_BASE}/triplets.jsonl"
OUTPUT_DIR = f"{DRIVE_BASE}/embedding_domain"

os.makedirs(OUTPUT_DIR, exist_ok=True)
assert os.path.exists(DATA_PATH), f"Upload triplets.jsonl to {DRIVE_BASE}/ first"
print(f"Data:   {DATA_PATH}")
print(f"Output: {OUTPUT_DIR}")

In [ ]:
import json
import torch
from datasets import Dataset
from sentence_transformers import (
    SentenceTransformer,
    SentenceTransformerTrainer,
    SentenceTransformerTrainingArguments,
)
from sentence_transformers.losses import MultipleNegativesRankingLoss
from sentence_transformers.evaluation import TripletEvaluator

MODEL_CHECKPOINT = "sentence-transformers/all-MiniLM-L6-v2"
BATCH_SIZE = 64
EPOCHS     = 3
LR         = 2e-5

print(f"CUDA: {torch.cuda.is_available()}")

## Data

In [ ]:
rows = []
skipped = 0
with open(DATA_PATH, encoding="utf-8") as f:
    for i, line in enumerate(f):
        try:
            rows.append(json.loads(line))
        except json.JSONDecodeError:
            skipped += 1

if skipped:
    print(f"Skipped {skipped} malformed lines")

split_idx = int(len(rows) * 0.9)
train_rows, val_rows = rows[:split_idx], rows[split_idx:]

# MNR chỉ cần anchor + positive — negative trong batch tự động
train_ds = Dataset.from_list([
    {"anchor": r["query"], "positive": r["positive"]}
    for r in train_rows
])

print(f"Train: {len(train_ds)} | Val: {len(val_rows)}")

## Evaluator

In [ ]:
evaluator = TripletEvaluator(
    anchors=  [r["query"]    for r in val_rows],
    positives=[r["positive"] for r in val_rows],
    negatives=[r["negative"] for r in val_rows],
    name="nfcorpus-val",
)

## Training

In [ ]:
model = SentenceTransformer(MODEL_CHECKPOINT)
loss  = MultipleNegativesRankingLoss(model=model)

args = SentenceTransformerTrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    learning_rate=LR,
    warmup_ratio=0.1,
    fp16=torch.cuda.is_available(),
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="nfcorpus-val_cosine_accuracy",
    logging_steps=200,
)

trainer = SentenceTransformerTrainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    loss=loss,
    evaluator=evaluator,
)

trainer.train()

## Evaluation & Save

In [ ]:
results = evaluator(model)
score = results.get("nfcorpus-val_cosine_accuracy", list(results.values())[0])
print(f"Triplet accuracy: {score:.4f}")

model.save(OUTPUT_DIR)

## Download

In [ ]:
import shutil
from google.colab import files

shutil.make_archive("/content/embedding_domain", "zip", OUTPUT_DIR)
print(f"Zip size: {os.path.getsize(chr(47)+chr(99)+chr(111)+chr(110)+chr(116)+chr(101)+chr(110)+chr(116)+chr(47)+chr(101)+chr(109)+chr(98)+chr(101)+chr(100)+chr(100)+chr(105)+chr(110)+chr(103)+chr(95)+chr(100)+chr(111)+chr(109)+chr(97)+chr(105)+chr(110)+chr(46)+chr(122)+chr(105)+chr(112)) / 1e6:.1f} MB")
files.download("/content/embedding_domain.zip")